# 04 – Kiểm tra L3 cho data gated (IEMOCAP, MER2023–2026)
Notebook này chỉ in **số liệu tổng hợp**: số mẫu, phân phối nhãn, tên cột, cấu trúc thư mục. Nó **không** in mẫu, transcript hay nhãn của từng câu, và không in mật khẩu giải nén, để tuân thủ điều khoản không phân phối lại data và annotation.

**Cài đặt notebook:** Accelerator = **None (CPU)**; **Internet = On**.
1. **IEMOCAP:** *Add Input* → dataset Kaggle (private) chứa bản IEMOCAP của bạn (thư mục có `Session1` … `Session5`). Notebook tự tìm trong `/kaggle/input`.
2. **MER:** *Add-ons → Secrets* → thêm secret tên `HF_TOKEN` (token Hugging Face quyền *read* của tài khoản đã được duyệt `MERChallenge/*`), rồi bật secret cho notebook.
3. *Save Version → Save & Run All*. Thời gian chạy khoảng 5–15 phút: notebook chỉ đọc file nhỏ, và liệt kê `video-labeled.zip` từ xa chứ không tải cả file.

**Khi xong:** tải `verify_report.md` trong *Output* rồi gửi lại cho Claude (file chỉ chứa số liệu tổng hợp).

In [ ]:
RUN_IEMOCAP = True
RUN_MER = True
KIN, KOUT, WORK = '/kaggle/input', '/kaggle/working', '/tmp'

In [ ]:
import os, subprocess
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
if RUN_MER:
    sh('pip -q install -U huggingface_hub', check=False)
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
args = ['python', f'{WORK}/icme/datasets/verify/verify_gated.py', '--out', f'{KOUT}/verify_report.md']
if RUN_IEMOCAP: args += ['--iemocap', KIN]
if RUN_MER: args += ['--mer']
subprocess.run(args, check=False)
print(open(f'{KOUT}/verify_report.md').read())